# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# Architecture:
# - API Wrapper: Manages model interaction and stream rendering across providers (OpenAI & Ollama).
# - Memory Management: Retains multi-turn dialogue state with strict role alternation.
# - Token Accounting: Measures prompt token consumption via local tokenization.
# - Context Optimization: Triggers LLM-based summarization to enforce a fixed token ceiling.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display
from openai import OpenAI
import tiktoken


In [ ]:
# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
TOKEN_LIMIT = 1000

BASE_URL_OLLAMA = "http://localhost:11434/v1"
API_KEY_OLLAMA = "OLLAMA_API_KEY"

In [ ]:
#providers

openai = OpenAI()
llama = OpenAI(base_url=BASE_URL_OLLAMA, api_key=API_KEY_OLLAMA)

In [ ]:
# set up environment
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY and OPENAI_API_KEY.startswith('sk-proj-') and len(OPENAI_API_KEY) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

In [ ]:
#prompts

system_prompt = """
You are an expert programming mentor and software engineer. Your goal is to help learners truly understand coding concepts and write clean, idiomatic code.

Guidelines:
- Explain complex technical concepts in plain, intuitive terms using real-world analogies where helpful.
- Keep answers focused, practical, and structured with clear markdown headings and bullet points.
- When providing code, write modern, readable code with brief comments explaining the 'why', not just the 'what'.
- Highlight common pitfalls and best practices relevant to the topic.
- Be encouraging, concise, and avoid unnecessary filler or generic introductions.
""".strip()

In [ ]:
def get_response_from_model(messages, model=MODEL_GPT):
    if model == MODEL_GPT:
        client = openai
    elif model == MODEL_LLAMA:
        client = llama
    else:
        raise ValueError("Unsupported model")

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        stream_options={"include_usage": True}
    )
    response = ""
    prompt_tokens = 0
    completion_tokens = 0
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id)
        if chunk.usage:
            prompt_tokens = chunk.usage.prompt_tokens
            completion_tokens = chunk.usage.completion_tokens
            total_tokens = chunk.usage.total_tokens
            print(f"\n[Tokens Used -> Prompt: {prompt_tokens}, Reply: {completion_tokens}, Total: {total_tokens}]")
    return response


In [ ]:
# Token Counting & Context Compacting Helpers

def count_tokens_for_messages(messages, model=MODEL_GPT):
    """Counts the total tokens in the messages history using tiktoken."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
        
    num_tokens = 0
    for message in messages:
        num_tokens += 4  # overhead for role and message wrappers
        for key, value in message.items():
            if value:
                num_tokens += len(encoding.encode(str(value)))
    num_tokens += 2  # priming tokens for assistant reply
    return num_tokens


def compact_messages(messages, model=MODEL_GPT):
    """
    Summarizes older conversation history to reduce token count
    while preserving the system prompt, core facts, and the latest user prompt.
    Uses the selected model for compaction.
    """
    print(f"\n⚠️ Context exceeded token limit! Compacting history using {model}...")
    
    # Select the correct client for the chosen model
    client = openai if model == MODEL_GPT else llama
    
    system_message = messages[0]
    older_history = messages[1:-1]    # Everything between system prompt and current question
    latest_user_message = messages[-1]  # The question the user just asked
    
    if not older_history:
        return messages

    # Format older history into a readable script
    formatted_history = "\n".join([f"{m['role'].capitalize()}: {m['content']}" for m in older_history])
    
    summarization_prompt = [
        {
            "role": "system", 
            "content": "You are a concise summarizer. Summarize this conversation history, capturing key topics, facts, and context established so far. Keep it concise (under 150 words)."
        },
        {
            "role": "user", 
            "content": formatted_history
        }
    ]
    
    # Use the selected model to summarize
    summary_res = client.chat.completions.create(
        model=model,
        messages=summarization_prompt
    )
    summary = summary_res.choices[0].message.content
    
    # Reconstruct messages: System Prompt + Summary + Latest Question
    compacted = [
        system_message,
        {"role": "system", "content": f"Summary of previous conversation:\n{summary}"},
        latest_user_message
    ]
    
    print(f"✅ Compaction complete! Tokens reduced to: {count_tokens_for_messages(compacted, model=model)}\n")
    return compacted


In [ ]:
# Chat loop with upfront model selection, token monitoring, and automatic compaction

TOKEN_LIMIT = 1000

# 1. Model Selection before starting the chat
print("Select a model for this chat session:")
print("1: GPT-4o-mini (OpenAI)")
print("2: Llama 3.2 (Local Ollama)")
choice = input("Enter 1 or 2 (default is 1): ").strip()

if choice == "2":
    selected_model = MODEL_LLAMA
else:
    selected_model = MODEL_GPT

print(f"\n🚀 Starting session with model: {selected_model}\n")

# 2. Initialize conversation messages
messages = [
    {"role": "system", "content": system_prompt},
]

# 3. Continuous Chat Loop
while True:
    user_prompt = input("Please enter your question for the AI assistant (or 'quit' to exit): ")
    if user_prompt.strip().lower() in ["quit", "exit", "q"]:
        print("Exiting chat. Goodbye!")
        break
    if not user_prompt.strip():
        continue

    print(f"\nUser prompt: {user_prompt}")
    messages.append({"role": "user", "content": user_prompt})

    # Calculate token count before API call
    current_tokens = count_tokens_for_messages(messages, model=selected_model)

    # Compact using the SAME selected model if limit is exceeded
    if current_tokens > TOKEN_LIMIT:
        messages = compact_messages(messages, model=selected_model)

    # Get assistant response using the selected model
    assistant_response = get_response_from_model(messages, model=selected_model)
    messages.append({"role": "assistant", "content": assistant_response})
